In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
TOPIC_DIR = PROJECT_ROOT / '02_Radiomics/02_Modeling_and_Evaluation'
EXTERNAL_INPUT_DIR = Path(os.environ.get('MMDLPC_EXTERNAL_INPUTS', str(DATA_ROOT / 'External_Inputs')))

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
sys.path.insert(0, str(PROJECT_ROOT))
from modeling_pipeline import (
    read_indexed, parse_p_value, correlation_keep_indices, make_pipeline,
    training_cv, best_finite_parameters, fit_on_training,
    model_estimators, training_search_grids, positive_probability,
)
DATA_DIR = DATA_ROOT / '00_Shared_Data_and_Code/Data'


# Radiomics feature extraction and modeling

Read raw features and the fixed patient partition. Estimate preprocessing and model parameters within the training data.


## 一、数据校验
首先需要检查诊断数据，如果显示`检查通过！`择可以正常运行之后的，否则请根据提示调整数据。

**注意**：这里要求images和masks文件夹中的文件名必须一一对应。e.g. `1.nii.gz`为images中的一个文件，在masks文件夹必须也存在一个`1.nii.gz`文件。

当然也可以使用自定义的函数，获取解析数据。

### 指定数据

此模块有3个需要自己定义的参数

1. `mydir`: 数据存放的路径。
2. `labelf`: 每个样本的标注信息文件。
3. `labels`: 要让AI系统学习的目标，例如肿瘤的良恶性、T-stage等。

In [ ]:
import os
from IPython.display import display
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

os.makedirs(str(TOPIC_DIR), exist_ok=True)
os.makedirs(str(TOPIC_DIR), exist_ok=True)
os.makedirs(str(TOPIC_DIR), exist_ok=True)
# 设置数据目录
# mydir = r'你自己数据的路径'
mydir = str(EXTERNAL_INPUT_DIR / "MRI_Images_and_Masks")
# 对应的标签文件
# labelf = r'你自己标注数据的文件地址'
labelf = str(DATA_ROOT / '00_Shared_Data_and_Code/Data/CPGEA-TCGA 20230106 OK.csv')
# 读取标签数据列名
labels = ['HRR_ANY']
# 前缀
prefix = 'Rad_'


### images和masks匹配

这里要求images和masks文件夹中的文件名必须一一对应。e.g. `1.nii.gz`为images中的一个文件，在masks文件夹必须也存在一个`1.nii.gz`文件。

当然也可以使用自定义的函数，获取解析数据。

In [ ]:
if not (DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_features_reference.csv').exists():
    from pathlib import Path
    from onekey_algo.custom.components.Radiology import diagnose_3d_image_mask_settings, get_image_mask_from_dir
    
    # 生成images和masks对，一对一的关系。也可以自定义替换。
    images, masks = get_image_mask_from_dir(mydir, images='images', masks='masks')
    
    # 自定义获取images和masks数据的方法，下面的例子为，每个样本一个文件夹，图像是以im.nii结尾，mask是以seg.nii结尾。
    # def get_images_mask(mydir):
    #     images = []
    #     masks = []
    #     for root, dirs, files in os.walk(mydir):
    #         for f in files:
    #             if f.endswith('im.nii'):
    #                 images.append(os.path.join(root, f))
    #             if f.endswith('seg.nii'):
    #                 masks.append(os.path.join(root, f))
    #     return images, masks
    # images, masks = get_images_mask(mydir)
    
    # diagnose_3d_image_mask_settings(images, masks, verbose=True)
    print(f'获取到{len(images)}个样本。')


# 传统组学特征

使用pyradiomics提取传统组学特征，正常这块不需要修改，下面是具体的Onekey封装的接口。

```python
def extract(self, images: Union[str, List[str]], 
            masks: Union[str, List[str]], labels: Union[int, List[int]] = 1, settings=None)
"""
    * images: List结构，待提取的图像列表。
    * masks: List结构，待提取的图像对应的mask，与Images必须一一对应。
    * labels: 提取标注为什么标签的特征。默认为提取label=1的。
    * settings: 其他提取特征的参数。默认为None。

"""
```

```python
def get_label_data_frame(self, label: int = 1, column_names=None, images='images', masks='labels')
"""
    * label: 获取对应label的特征。
    * columns_names: 默认为None，使用程序设定的列名即可。
"""
```
    
```python
def get_image_mask_from_dir(root, images='images', masks='labels')
"""
    * root: 待提取特征的目录。
    * images: root目录中原始数据的文件夹名。
    * masks: root目录中标注数据的文件夹名。
"""
```


In [ ]:
import pandas as pd
 


if os.path.exists(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_features_reference.csv')):
    rad_data = pd.read_csv(str(DATA_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_features_reference.csv'), header=0)
else:
    from onekey_algo.custom.components.Radiology import ConventionalRadiomics
    # 如果要自定义一些特征提取方式，可以使用param_file。
    param_file = str(PROJECT_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/radiomics_extraction_MR_3mm.yaml')
#     param_file = None
    radiomics = ConventionalRadiomics(param_file, correctMask=True)
    radiomics.extract(images, masks)
    rad_data = radiomics.get_label_data_frame(label=1)
    rad_data.columns = [c.replace('-', '_') for c in rad_data.columns]
    rad_data.to_csv(str(PROJECT_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/rad_features_reference.csv'), header=True, index=False)
rad_data


In [ ]:
series = []
for ending, suffix in [('-1.nii.gz', 'DWI'), ('-2.nii.gz', 'T2WI')]:
    frame = rad_data.loc[rad_data['ID'].str.endswith(ending)].copy()
    frame['ID'] = frame['ID'].str.removesuffix(ending)
    if frame.ID.isna().any() or frame.ID.duplicated().any():
        raise ValueError('Duplicate or missing patient/sequence IDs.')
    series.append(frame.set_index('ID').add_suffix(suffix))
rad_data = series[0].join(series[1], how='inner', validate='one_to_one').reset_index()


## 特征统计

In [ ]:
import matplotlib.pyplot as plt
sorted_counts = pd.DataFrame([c.split('_')[-2] for c in rad_data.columns if c !='ID']).value_counts()
plt.pie(sorted_counts, labels=[i[0] for i in sorted_counts.index], startangle=0,
        counterclock = False, autopct = '%.1f%%')
display(sorted_counts)
plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}feature_ratio.svg'), bbox_inches = 'tight')
plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}feature_ratio.pdf'), bbox_inches = 'tight')


## 标注数据

数据以csv格式进行存储，这里如果是其他格式，可以使用自定义函数读取出每个样本的结果。

要求label_data为一个`DataFrame`格式，包括ID列以及后续的labels列，可以是多列，支持Multi-Task。

In [ ]:
radiology_features = rad_data.set_index('ID')


## Align features, labels and the fixed patient partition


In [ ]:
partition = read_indexed(DATA_DIR / 'R_fixed_partition.csv')
if set(partition.Split) != {'Train', 'Test'}:
    raise ValueError('Expected a pre-specified Train/Test partition.')
label_table = read_indexed(DATA_DIR / 'CPGEA-TCGA 20230106 OK.csv')
if not partition.index.isin(radiology_features.index).all() or not partition.index.isin(label_table.index).all():
    raise ValueError('The feature or label table is missing patients in the partition.')
raw_features = radiology_features.loc[partition.index].apply(pd.to_numeric, errors='raise')
raw_features = raw_features.replace([np.inf, -np.inf], np.nan)
X_source = raw_features.add_prefix('radiology__')
y_source = label_table.loc[partition.index, 'HRR_ANY']
if not y_source.isin([0, 1]).all():
    raise ValueError('Expected observed binary HRR_ANY labels.')
y_source = y_source.astype(int)
split = partition.Split
train_rows = split.eq('Train')
test_rows = split.eq('Test')
train_ids = X_source.index[train_rows]
test_ids = X_source.index[test_rows]
if not X_source.index.is_unique or not set(train_ids).isdisjoint(test_ids):
    raise ValueError('Patient IDs must be unique and the partition must be disjoint.')
labels = ['HRR_ANY']
label_data = label_table.loc[partition.index, labels + ['group']].reset_index()
ids = pd.Series(partition.index, index=partition.index, name='ID')

combined_data = raw_features.join(label_table.loc[partition.index, labels + ['group']], validate='one_to_one')


## 获取到数据的统计信息

1. count，统计样本个数。
2. mean、std, 对应特征的均值、方差
3. min, 25%, 50%, 75%, max，对应特征的最小值，25,50,75分位数，最大值。

In [ ]:
combined_data.describe()


## Training-fitted imputation and standardization for feature diagnostics


In [ ]:
observed_columns = raw_features.columns[raw_features.loc[train_rows].notna().any()]
imputer = SimpleImputer(strategy='median').fit(raw_features.loc[train_rows, observed_columns])
filled = imputer.transform(raw_features[observed_columns])
scaler = StandardScaler().fit(filled[train_rows.to_numpy()])
data = pd.DataFrame(scaler.transform(filled), index=partition.index, columns=observed_columns)
data = data.join(label_table.loc[partition.index, labels + ['group']], validate='one_to_one')
data.describe()


In [ ]:
feature_columns = [column for column in data if column not in labels + ['group']]


## Training-only Welch tests for feature diagnostics


In [ ]:
import seaborn as sns
from scipy.stats import ttest_ind
training_features = data.loc[train_rows, feature_columns]
training_labels = y_source.loc[train_rows]
feature_tests = ttest_ind(training_features.loc[training_labels.eq(0)],
                         training_features.loc[training_labels.eq(1)], axis=0, equal_var=False)
stats = pd.DataFrame({'feature_name': feature_columns, 'pvalue': feature_tests.pvalue})
stats


#### 输出特征分布的图

In [ ]:
import matplotlib.pyplot as plt

def map2float(x):
    return parse_p_value(x)

stats[['pvalue']] = stats[['pvalue']].applymap(map2float)
stats[['group']] = stats[['feature_name']].applymap(lambda x: x.split('_')[-2])
stats = stats[['feature_name', 'pvalue', 'group']]
g = sns.catplot(x="group", y="pvalue", data=stats, kind="violin")
g.fig.set_size_inches(15,10)
sns.stripplot(x="group", y="pvalue", data=stats, ax=g.ax, color='black')
plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}feature_stats.svg'), bbox_inches = 'tight')
plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}feature_stats.pdf'), bbox_inches = 'tight')


### Apply the diagnostic p-value threshold


In [ ]:
pvalue = 0.05
sel_feature = stats.loc[stats.pvalue.lt(pvalue), 'feature_name'].tolist()
if not sel_feature:
    raise ValueError('No features passed training-only statistical screening.')


### 相关系数

计算相关系数的方法有3种可供选择
1. pearson （皮尔逊相关系数）: standard correlation coefficient

2. kendall (肯德尔相关性系数) : Kendall Tau correlation coefficient

3. spearman (斯皮尔曼相关性系数): Spearman rank correlation

三种相关系数参考：https://blog.csdn.net/zmqsdu9001/article/details/82840332

In [ ]:
pearson_corr = data.loc[train_rows, sel_feature].corr('pearson')


### 相关系数可视化

通过修改变量名，可以可视化不同相关系数下的相关矩阵。

**注意**：当特征特别多的时候（大于100），尽量不要可视化，否则运行时间会特别长。

In [ ]:
pearson_corr


### 聚类分析

通过修改变量名，可以可视化不同相关系数下的相聚类分析矩阵。

注意：当特征特别多的时候（大于100），尽量不要可视化，否则运行时间会特别长。

### Training-only correlation filtering


In [ ]:
positions = correlation_keep_indices(data.loc[train_rows, sel_feature].to_numpy(), 'pearson', .9)
sel_feature = [sel_feature[i] for i in positions]


### 过滤特征

通过`sel_feature`过滤出筛选出来的特征。

In [ ]:
sel_data = data[sel_feature + labels + ['group']]
sel_data.describe()


### Selected-feature test results


In [ ]:
stats.loc[stats.feature_name.isin(sel_feature)].sort_values('pvalue')


## Read the fixed training and test partitions


In [ ]:
X_data = X_source.loc[train_rows].copy()
X_test_data = X_source.loc[test_rows].copy()
y_data = y_source.loc[train_rows].to_frame('HRR_ANY')
y_test_data = y_source.loc[test_rows].to_frame('HRR_ANY')
n_classes = 2


### Cross-validation of preprocessing and LASSO within the training set


In [ ]:
selection_classifier = LogisticRegression(max_iter=1000, random_state=0)
selection_grid = training_search_grids('R', {'LR': selection_classifier})['LR']
selection_grid['features__R__pvalue_threshold'] = [0.05]
selection_search = GridSearchCV(
    make_pipeline(X_data, 'R', selection_classifier), selection_grid,
    scoring='roc_auc', cv=training_cv(y_data['HRR_ANY']),
    n_jobs=1, error_score=np.nan, refit=False)
selection_search.fit(X_data, y_data['HRR_ANY'])
selection_parameters = best_finite_parameters(selection_search)
selection_pipeline = make_pipeline(X_data, 'R', selection_classifier)
selection_pipeline.set_params(**selection_parameters).fit(X_data, y_data['HRR_ANY'])
selector = selection_pipeline['features'].named_transformers_['R']
alpha = selector.alpha


### Training cross-validation AUC by LASSO alpha


In [ ]:
selection_results = pd.DataFrame(selection_search.cv_results_)
alpha_key = next(key for key in selection_results if key.endswith('__alpha'))
plt.figure(figsize=(6, 4))
plt.errorbar(selection_results[alpha_key].astype(float), selection_results['mean_test_score'],
             yerr=selection_results['std_test_score'], marker='o', markersize=3)
plt.xscale('log')
plt.xlabel('LASSO alpha')
plt.ylabel('Training cross-validation AUC (mean and SD)')
plt.tight_layout()
plt.savefig(TOPIC_DIR / 'Reference_Rad_feature_selection_CV_AUC.pdf', bbox_inches='tight')


### Fitted LASSO coefficients


In [ ]:
models = [selector.lasso_]
column_names = np.asarray(selector.observed_columns_)[selector.correlation_indices_]


### Features with nonzero LASSO coefficients


In [ ]:
selected_features = [selector.selected_features_.tolist()]
feat_coef = [(name, coefficient) for name, coefficient in zip(column_names, selector.lasso_.coef_)
             if abs(coefficient) > 1e-6]
feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
feat_coef_df


### 特征权重

In [ ]:
feat_coef = sorted(feat_coef, key=lambda x: x[1])
feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
feat_coef_df.plot(x='feature_name', y='Coefficients', kind='barh')

plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}feature_weights.svg'), bbox_inches = 'tight')
plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}feature_weights.pdf'), bbox_inches = 'tight')


### Export training-fitted features for inspection

Model fitting below starts from raw features and refits preprocessing in each training fold.


In [ ]:
selected_values = pd.DataFrame(selection_pipeline['features'].transform(X_source),
    index=X_source.index, columns=selection_pipeline['features'].get_feature_names_out())
selected_values.index.name = 'ID'
selected_values.reset_index().to_csv(TOPIC_DIR / 'rad_sel_features_reference.csv', index=False)
selected_values.columns


## Candidate classifiers


In [ ]:
models = model_estimators('R')
model_names = list(models)


### Nested cross-validation within the fixed training partition


In [ ]:
import seaborn as sns
search_grids = training_search_grids('R', models)
for grid in search_grids.values():
    grid['features__R__pvalue_threshold'] = [0.05]
fitted_models = {}
cv_tables = []
parameter_tables = {}
for name, classifier in models.items():
    fitted, cv_result, parameters = fit_on_training(
        X_data, y_data['HRR_ANY'], 'R', classifier, search_grids[name])
    fitted_models[name] = fitted
    parameter_tables[name] = parameters
    cv_result['Model'] = name
    cv_tables.append(cv_result)
cv_results = pd.concat(cv_tables, ignore_index=True)
X_train_sel, X_test_sel = X_data, X_test_data
y_train_sel, y_test_sel = y_data, y_test_data
plt.figure(figsize=(9, 4))
sns.boxplot(data=cv_results, x='Model', y='AUC', order=model_names)
plt.ylabel('Training nested cross-validation AUC')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(TOPIC_DIR / 'Reference_Rad_model_cv.pdf', bbox_inches='tight')


## Save models fitted on the training partition


In [ ]:
import joblib
targets = [[fitted_models[name] for name in model_names]]
TOPIC_DIR.mkdir(parents=True, exist_ok=True)
for name, fitted in fitted_models.items():
    joblib.dump(fitted, TOPIC_DIR / f'Reference_Rad_{name}_HRR_ANY.pkl')
    estimator = fitted['classifier']
    if hasattr(estimator, 'named_steps') and 'estimator' in estimator.named_steps:
        estimator = estimator.named_steps['estimator']
    feature_names = fitted['features'].get_feature_names_out()
    importance = getattr(estimator, 'feature_importances_', None)
    if importance is None and hasattr(estimator, 'coef_'):
        importance = np.ravel(estimator.coef_)
    if importance is not None:
        table = pd.DataFrame({'Feature': feature_names, 'Importance': importance}).sort_values('Importance')
        table.plot.barh(x='Feature', y='Importance', figsize=(7, max(4, len(table) * .15)), legend=False)
        plt.tight_layout()
        plt.savefig(TOPIC_DIR / f'Reference_Rad_{name}_feature_importance.pdf', bbox_inches='tight')
        plt.close()


## 预测结果

* predictions，二维数据，每个label对应的每个模型的预测结果。
* pred_scores，二维数据，每个label对应的每个模型的预测概率值。

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from onekey_algo.custom.components.delong import calc_95_CI
from onekey_algo.custom.components.metrics import analysis_pred_binary

import onekey_algo.custom.components as okcomp

metric = []
pred_sel_idx = []
predictions = [[(model.predict(X_train_sel), model.predict(X_test_sel))  
                for model in target] for label, target in zip(labels, targets)]
pred_scores = [[(model.predict_proba(X_train_sel), model.predict_proba(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]
for label, prediction, scores in zip(labels, predictions, pred_scores):
    pred_sel_idx_label = []
    for mname, (train_pred, test_pred), (train_score, test_score) in zip(model_names, prediction, scores):
        # 计算训练集指数
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_train_sel[label], 
                                                                                              train_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"Train"))

        # 计算验证集指标
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_test_sel[label], 
                                                                                              test_score[:, 1], use_youden=True)
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, 'Test'))
        # 计算thres对应的sel idx
        pred_sel_idx_label.append(np.logical_or(test_score[:, 0] >= thres, test_score[:, 1] >= thres))

    pred_sel_idx.append(pred_sel_idx_label)
metric = pd.DataFrame(metric, index=None, columns=['model_name', 'Accuracy', 'AUC', '95% CI',
                                                   'Sensitivity', 'Specificity', 
                                                   'PPV', 'NPV', 'Precision', 'Recall', 'F1',
                                                   'Threshold', 'Task'])
metric


### 绘制曲线

绘制的不同模型的准确率柱状图和折线图曲线。

In [ ]:
import seaborn as sns

plt.subplot(211)
sns.barplot(x='model_name', y='Accuracy', data=metric, hue='Task')
plt.subplot(212)
sns.lineplot(x='model_name', y='Accuracy', data=metric, hue='Task')
plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_acc.svg'), bbox_inches = 'tight')
plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_acc.pdf'), bbox_inches = 'tight')


### 绘制ROC曲线
确定最好的模型，并且绘制曲线。

```python
def draw_roc(y_test, y_score, title='ROC', labels=None):
```

`sel_model = ['SVM', 'KNN']`参数为想要绘制的模型对应的参数。

In [ ]:
sel_model = model_names

for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
    
        # Plot all ROC curves
        plt.figure(figsize=(8, 8))
        for pred_score, label in zip(pred_scores, labels):
            okcomp.comp1.draw_roc([np.array(y_train_sel[label]), np.array(y_test_sel[label])], 
                                  pred_score[sel_model_idx], 
                                  labels=['Train', 'Test'], title=f"Model: {sm}")
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_{sm}_roc.svg'), bbox_inches = 'tight')
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_{sm}_roc.pdf'), bbox_inches = 'tight')


#### 汇总所有模型

In [ ]:
sel_model = model_names

for pred_score, label in zip(pred_scores, labels):
    pred_val_scores = []
    pred_test_scores = []
    
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            pred_test_scores.append(pred_score[sel_model_idx][1])
    okcomp.comp1.draw_roc([np.array(y_test_sel[label])] * len(pred_test_scores), 
                          pred_test_scores, 
                          labels=sel_model, title=f"Model AUC")
    plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_roc.svg'), bbox_inches = 'tight')
    plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_roc.pdf'), bbox_inches = 'tight')


### DCA 决策曲线

In [ ]:
from onekey_algo.custom.components.comp1 import plot_DCA

for pred_score, label in zip(pred_scores, labels):
    pred_test_scores = []
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            okcomp.comp1.plot_DCA(pred_score[sel_model_idx][1][:,1], np.array(y_test_sel[label]),
                                  title=f'Model {sm} DCA')
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_{sm}_dca.svg'), bbox_inches = 'tight')
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_{sm}_dca.pdf'), bbox_inches = 'tight')


### 绘制混淆矩阵

绘制混淆矩阵，[混淆矩阵解释](https://baike.baidu.com/item/%E6%B7%B7%E6%B7%86%E7%9F%A9%E9%98%B5/10087822?fr=aladdin)
`sel_model = ['SVM', 'KNN']`参数为想要绘制的模型对应的参数。

如果需要修改标签到名称的映射，修改`class_mapping={1:'1', 0:'0'}`

In [ ]:
# 设置绘制参数
sel_model = model_names
c_matrix = {}

for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
        for idx, label in enumerate(labels):
            cm = okcomp.comp1.calc_confusion_matrix(predictions[idx][sel_model_idx][1], y_test_sel[label],
#                                                     sel_idx = pred_sel_idx[idx][sel_model_idx],
                                                    class_mapping={1:'1', 0:'0'}, num_classes=2)
            c_matrix[label] = cm
            plt.figure(figsize=(5, 4))
            plt.title(f'Model:{sm}')
            okcomp.comp1.draw_matrix(cm, norm=False, annot=True, cmap='Blues', fmt='.0f')
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_{sm}_cm.svg'), bbox_inches = 'tight')
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_{sm}_cm.pdf'), bbox_inches = 'tight')


### 样本预测直方图

绘制每个样本的预测结果以及对应的真实结果, 图例中label=xx可以修改成自己类别的真实标签。

In [ ]:
sel_model = model_names
c_matrix = {}

for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
        for idx, label in enumerate(labels):            
            okcomp.comp1.draw_predict_score(pred_scores[idx][sel_model_idx][1], y_test_sel[label])
            plt.title(f'{sm} test sample predict score')
            plt.legend(labels=["label=0", "label=1"],loc="lower right") 
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_{sm}_sample_dis.svg'), bbox_inches = 'tight')
            plt.savefig(str(TOPIC_DIR / f'Reference_{prefix}model_test_{sm}_sample_dis.pdf'), bbox_inches = 'tight')
            plt.show()


## 保存模型结果

可以把模型预测的标签结果以及每个类别的概率都保存下来。

In [ ]:
import os
import numpy as np

os.makedirs(str(TOPIC_DIR), exist_ok=True)

for idx, label in enumerate(labels):
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            target = targets[idx][sel_model_idx]
            # 预测训练集和测试集数据。
            train_indexes = np.reshape(np.array(ids.loc[X_train_sel.index]), (-1, 1)).astype(str)
            test_indexes = np.reshape(np.array(ids.loc[X_test_sel.index]), (-1, 1)).astype(str)
            y_train_pred_scores = target.predict_proba(X_train_sel)
            y_test_pred_scores = target.predict_proba(X_test_sel)
            columns = ['ID'] + [f"{label}-{i}"for i in range(y_test_pred_scores.shape[1])]
            # 保存预测的训练集和测试集结果
            result_train = pd.DataFrame(np.concatenate([train_indexes, y_train_pred_scores], axis=1), columns=columns)
            result_train.to_csv(str(TOPIC_DIR / f'Reference_{prefix}{sm}_train.csv'), index=False)
            result_test = pd.DataFrame(np.concatenate([test_indexes, y_test_pred_scores], axis=1), columns=columns)
            result_test.to_csv(str(TOPIC_DIR / f'Reference_{prefix}{sm}_test.csv'), index=False)


In [ ]:
import pandas as pd

train = pd.read_csv(str(TOPIC_DIR / 'Reference_Rad_XGBoost_train.csv'))
train['group'] = 'train'
val = pd.read_csv(str(TOPIC_DIR / 'Reference_Rad_XGBoost_test.csv'))
val['group'] = 'test'

data = pd.concat([train, val], axis=0)
data['ID'] = data['ID'].astype(str)
data = pd.merge(data[['ID', 'group']], label_data[['ID'] + labels], on='ID', how='inner')
data.to_csv(str(PROJECT_ROOT / '02_Radiomics/02_Modeling_and_Evaluation/radiomics_group_reference.csv'), index=False)


In [ ]:
pd.concat([train, val], axis=0)
